# 10 · Stay-point có đúng không?

GeoLife **không có nhãn chỗ ở**, nên không có đáp án để so trực tiếp. Việc kiểm chứng chia thành bốn tầng, mỗi tầng trả lời một câu hỏi khác nhau:

| Tầng | Câu hỏi | Ở đâu |
|---|---|---|
| 1 | Code có làm đúng như định nghĩa không? | Unit test trên quỹ đạo giả biết trước đáp án (`tests/unit/test_features/test_stay_point.py`) |
| 2 | Định nghĩa có khớp thực tế không? | So với nhãn chuyến đi: stay-point giả trong chuyến xe, độ phủ các khoảng dừng, sai lệch mốc thời gian (notebook 07) |
| **3** | **Trên toàn bộ dữ liệu thật, mọi stay-point có thoả các tính chất bắt buộc không?** | **Notebook này, mục 1–2** |
| 4 | Nhìn trên bản đồ, stay-point có hợp lý không? | Chấm một mẫu ngẫu nhiên có phân tầng (mục 3, làm sau) |

**Tầng 3** chạy bộ phát hiện của `src/` trên điểm sạch của cả 182 user (`data/processed/users`), rồi kiểm từng stay-point với chính các điểm GPS của nó. Mỗi tính chất là hệ quả trực tiếp của định nghĩa, nên **chỉ cần một vi phạm là có lỗi**. Đây không phải phép đo xác suất.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "10_staypoint_validation"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
from gps.data.processing import _haversine_vectorized
from gps.features.stay_point import MAX_GAP_HOURS, StayPointDetector

USERS_DIR = Path("../data/processed/users")
detector = StayPointDetector()            # mặc định của src: CENTROID, 200 m, 30 phút, cắt tại khoảng ngắt > 18 h
RADIUS_M = detector.distance_threshold_m
MIN_MINUTES = detector.time_threshold / pd.Timedelta(minutes=1)
UNOBSERVED_S = detector.unobserved_gap.total_seconds()
STAY_CACHE = CACHE_DIR / "stays"
STAY_CACHE.mkdir(exist_ok=True)

## 1. Stay-point của 182 user
Một lần chạy toàn bộ mất khoảng 1 giờ. Kết quả được cache theo từng user; chạy lại sẽ tiếp tục từ user chưa xong.

In [2]:
users = sorted(p.stem.split("_")[1] for p in USERS_DIR.glob("user_*.parquet"))
for k, uid in enumerate(users, start=1):
    path = STAY_CACHE / f"user_{uid}.parquet"
    if path.exists():
        continue
    print(f"\r[{k}/{len(users)}] user {uid}", end="", flush=True)
    df = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["datetime", "lat", "lon", "altitude_m"])
    pd.DataFrame([{"user_id": uid, "arrival": pd.Timestamp(sp.arrival_time), "departure": pd.Timestamp(sp.departure_time),
                   "lat": sp.lat, "lon": sp.lon, "duration_min": sp.duration_minutes, "observed_min": sp.observed_minutes,
                   "num_points": sp.num_points} for sp in detector.detect(df)],
                 columns=["user_id", "arrival", "departure", "lat", "lon", "duration_min", "observed_min", "num_points"]
                 ).astype({"lat": float, "lon": float, "duration_min": float, "observed_min": float}).to_parquet(path)
print()
stays = pd.concat([d for p in sorted(STAY_CACHE.glob("user_*.parquet")) if len(d := pd.read_parquet(p))], ignore_index=True)
print(f"Stay-points: {len(stays):,} from {stays['user_id'].nunique()} of {len(users)} users")

Stay-points: 18,597 from 174 of 182 users


## 2. Các tính chất bắt buộc
Với mỗi stay-point, lấy lại các điểm GPS sạch của user nằm trong khoảng [giờ đến, giờ đi], rồi kiểm:

| Tính chất | Vì sao phải đúng |
|---|---|
| `bounds_are_points` | Giờ đến và giờ đi là thời điểm của điểm GPS thật, không phải giá trị nội suy |
| `num_points_match` | Số điểm báo cáo bằng số điểm thật trong khoảng thời gian đó |
| `centroid_is_mean` | Tâm là trung bình toạ độ các điểm (lệch < 1 cm) |
| `duration_ok` | Thời lượng ≥ 30 phút và bằng giờ đi trừ giờ đến |
| `observed_ok` | `observed_minutes` tính lại được từ các bước ≤ 1200 s, và ≤ thời lượng |
| `no_gap_over_max` | Không có khoảng ngắt > 18 h giữa hai điểm liên tiếp bên trong |
| `running_centroid_ok` | Mỗi điểm cách **tâm tại lúc nó được thêm vào** ≤ 200 m (đúng quy tắc CENTROID) |
| `window_is_maximal` | Điểm ngay sau cửa sổ (nếu có) cách tâm cuối > 200 m hoặc nằm sau một khoảng ngắt > 18 h, tức cửa sổ không thể kéo dài thêm |
| `no_overlap` | Các stay-point của một user không chồng thời gian, và có thứ tự |

In [3]:
def check_user(uid: str, s: pd.DataFrame) -> pd.DataFrame:
    p = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["datetime", "lat", "lon"]).sort_values("datetime", kind="stable")
    ts, la, lo = p["datetime"].to_numpy(), p["lat"].to_numpy(), p["lon"].to_numpy()
    max_gap = np.timedelta64(int(MAX_GAP_HOURS * 3600), "s")
    s = s.sort_values("arrival")
    out = []
    for r in s.itertuples():
        i0 = np.searchsorted(ts, np.datetime64(r.arrival), "left")
        i1 = np.searchsorted(ts, np.datetime64(r.departure), "right") - 1
        wt, wl, wo = ts[i0:i1 + 1], la[i0:i1 + 1], lo[i0:i1 + 1]
        n = len(wt)
        steps = np.diff(wt) / np.timedelta64(1, "s")
        clat, clon = wl.mean(), wo.mean()
        # tâm tại lúc điểm k được thêm vào = trung bình các điểm 0..k-1
        run_lat, run_lon = np.cumsum(wl)[:-1] / np.arange(1, n), np.cumsum(wo)[:-1] / np.arange(1, n)
        run_d = _haversine_vectorized(wl[1:], wo[1:], run_lat, run_lon) if n > 1 else np.zeros(0)
        nxt = i1 + 1
        maximal = (nxt >= len(ts) or ts[nxt] - ts[i1] > max_gap
                   or _haversine_vectorized(la[nxt:nxt + 1], lo[nxt:nxt + 1], np.array([clat]), np.array([clon]))[0] > RADIUS_M)
        observed = steps[steps <= UNOBSERVED_S].sum() / 60
        out.append({
            "user_id": uid, "arrival": r.arrival, "duration_min": r.duration_min, "observed_min": r.observed_min,
            "num_points": r.num_points, "max_run_dist_m": run_d.max() if len(run_d) else 0.0,
            "bounds_are_points": n > 0 and wt[0] == np.datetime64(r.arrival) and wt[-1] == np.datetime64(r.departure),
            "num_points_match": n == r.num_points,
            "centroid_is_mean": _haversine_vectorized(np.array([clat]), np.array([clon]), np.array([r.lat]), np.array([r.lon]))[0] < 0.01,
            "duration_ok": r.duration_min >= MIN_MINUTES and abs((r.departure - r.arrival).total_seconds() / 60 - r.duration_min) < 1e-6,
            "observed_ok": abs(observed - r.observed_min) < 1e-6 and 0 <= r.observed_min <= r.duration_min + 1e-9,
            "no_gap_over_max": n < 2 or steps.max() <= MAX_GAP_HOURS * 3600,
            "running_centroid_ok": bool((run_d <= RADIUS_M + 1e-6).all()),
            "window_is_maximal": bool(maximal),
        })
    out = pd.DataFrame(out)
    out["no_overlap"] = np.r_[True, s["arrival"].to_numpy()[1:] > s["departure"].to_numpy()[:-1]]
    return out


checks = pd.concat([check_user(uid, s) for uid, s in stays.groupby("user_id")], ignore_index=True)
PROPERTIES = ["bounds_are_points", "num_points_match", "centroid_is_mean", "duration_ok", "observed_ok",
              "no_gap_over_max", "running_centroid_ok", "window_is_maximal", "no_overlap"]
violations = pd.DataFrame({"checked": len(checks), "violations": [int((~checks[c]).sum()) for c in PROPERTIES]}, index=PROPERTIES)
violations.to_csv(TABLE_DIR / "invariant_violations.csv")
print(f"Largest distance of a point to the running centroid when it joined: {checks['max_run_dist_m'].max():.1f} m")
violations

Largest distance of a point to the running centroid when it joined: 200.0 m


,checked,violations
bounds_are_points,18597,0
num_points_match,18597,0
centroid_is_mean,18597,0
duration_ok,18597,0
observed_ok,18597,0
no_gap_over_max,18597,0
running_centroid_ok,18597,0
window_is_maximal,18597,0
no_overlap,18597,0


In [4]:
bad = checks[~checks[PROPERTIES].all(axis=1)]
bad.head(20) if len(bad) else "No stay-point violates any property."

'No stay-point violates any property.'

### 2.1 Đối chứng: bộ kiểm tra có bắt được lỗi không?
"0 vi phạm" chỉ có ý nghĩa nếu bộ kiểm tra **có khả năng** báo lỗi. Ở đây ta cố ý làm hỏng stay-point của vài user theo từng cách, mỗi cách nhắm vào một tính chất, rồi đếm xem bộ kiểm tra bắt được bao nhiêu:
- **sửa trực tiếp kết quả**: dịch tâm 1 m, tăng số điểm, lùi giờ đến 1 giây, cộng 1 phút vào `observed_minutes` hoặc vào thời lượng, nhân đôi một dòng;
- **chạy detector với thông số khác** rồi kiểm theo thông số chuẩn: bán kính 300 m (điểm vượt 200 m so với tâm), bán kính 150 m (cửa sổ dừng quá sớm), không cắt khoảng ngắt (vắt qua khoảng ngắt > 18 h; user 028 và 140 có trường hợp này).

Với cách "chạy detector khác", chỉ những stay-point **khác** với kết quả chuẩn mới được tính.

In [5]:
MUT_USERS = ["010", "028", "041", "084", "140"]
base = stays[stays["user_id"].isin(MUT_USERS)]
key = lambda d: d["user_id"] + "|" + d["arrival"].astype(str) + "|" + d["departure"].astype(str)


def detect_with(det: StayPointDetector) -> pd.DataFrame:
    rows = []
    for uid in MUT_USERS:
        df = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["datetime", "lat", "lon"])
        rows += [{"user_id": uid, "arrival": pd.Timestamp(sp.arrival_time), "departure": pd.Timestamp(sp.departure_time),
                  "lat": sp.lat, "lon": sp.lon, "duration_min": sp.duration_minutes, "observed_min": sp.observed_minutes,
                  "num_points": sp.num_points} for sp in det.detect(df)]
    out = pd.DataFrame(rows)
    return out[~key(out).isin(set(key(base)))]          # chỉ giữ stay-point khác kết quả chuẩn


MUTATIONS = {
    "centroid moved 1 m": (base.assign(lat=base["lat"] + 1 / 111_320), "centroid_is_mean"),
    "num_points + 1": (base.assign(num_points=base["num_points"] + 1), "num_points_match"),
    "arrival 1 s earlier": (base.assign(arrival=base["arrival"] - pd.Timedelta(seconds=1)), "bounds_are_points"),
    "observed + 1 min": (base.assign(observed_min=base["observed_min"] + 1), "observed_ok"),
    "duration + 1 min": (base.assign(duration_min=base["duration_min"] + 1), "duration_ok"),
    "one row duplicated": (pd.concat([base, base.iloc[[0]]], ignore_index=True), "no_overlap"),
    "detector radius 300 m": (detect_with(StayPointDetector(30, 300)), "running_centroid_ok"),
    "detector radius 150 m": (detect_with(StayPointDetector(30, 150)), "window_is_maximal"),
    "no cut at gaps > 18 h": (detect_with(StayPointDetector(max_gap_hours=None)), "no_gap_over_max"),
}
rows = []
for name, (mutated, target) in MUTATIONS.items():
    c = pd.concat([check_user(uid, s) for uid, s in mutated.groupby("user_id")], ignore_index=True)
    rows.append({"mutation": name, "target_property": target, "stays_checked": len(c),
                 "caught_by_target": int((~c[target]).sum()), "caught_by_any": int((~c[PROPERTIES].all(axis=1)).sum())})
mutation_check = pd.DataFrame(rows)
mutation_check.to_csv(TABLE_DIR / "mutation_check.csv", index=False)
mutation_check

,mutation,target_property,stays_checked,caught_by_target,caught_by_any
0,centroid moved 1 m,centroid_is_mean,886,886,886
1,num_points + 1,num_points_match,886,886,886
2,arrival 1 s earlier,bounds_are_points,886,733,886
3,observed + 1 min,observed_ok,886,886,886
4,duration + 1 min,duration_ok,886,886,886
5,one row duplicated,no_overlap,887,1,1
6,detector radius 300 m,running_centroid_ok,1017,1010,1010
7,detector radius 150 m,window_is_maximal,753,646,646
8,no cut at gaps > 18 h,no_gap_over_max,42,42,42


## 3. Tầng 4: chấm bằng mắt một mẫu ngẫu nhiên
Tầng 3 không trả lời được câu hỏi "một cửa sổ hợp lệ có thật sự là lúc **ở yên một chỗ** không?" (ví dụ kẹt xe 40 phút trong bán kính 200 m vẫn thoả mọi tính chất). Câu hỏi này cần mắt người. Để kết quả là một con số có khoảng tin cậy, không phải "nhìn thấy ổn", mẫu được rút như sau (seed cố định, tái lập được):

| Nhóm | Số lượng | Định nghĩa | Dùng để |
|---|---|---|---|
| `random` | 30 | Ngẫu nhiên đều trên cả 18,597 stay-point | Ước lượng **độ chính xác chung** |
| `short` | 6 | Dài 30–40 phút | Rủi ro: đi chậm, kẹt xe |
| `long` | 6 | Dài > 12 h | Rủi ro: gộp nhiều lần ghi |
| `sparse` | 6 | `observed` < 10% thời lượng | Rủi ro: chủ yếu là thời gian không có điểm |
| `overnight` | 6 | Có mặt lúc 03:00 giờ địa phương | Quan trọng cho Home |
| `in_vehicle_label` | 6 | ≥ 80% thời gian nằm trong một chuyến xe có nhãn | Các stay-point "giả" theo notebook 07: thật sự sai, hay nhãn sai? |

Người chấm **không thấy nhóm** và thứ tự bị xáo trộn. Mỗi stay-point có các điểm GPS từ 30 phút trước giờ đến tới 30 phút sau giờ đi (tối đa 900 điểm, thưa đều nếu nhiều hơn), toạ độ quy về mét quanh tâm, cùng các chuyến đi có nhãn chồng lên khoảng đó.

In [6]:
import json

AUDIT_JSON = CACHE_DIR / "audit_sample.json"
VEHICLES = ["bus", "car", "taxi", "subway", "train"]
STRATA = {"random": 30, "short": 6, "long": 6, "sparse": 6, "overnight": 6, "in_vehicle_label": 6}
CONTEXT = pd.Timedelta(minutes=30)
MAX_POINTS = 900
rng = np.random.default_rng(20261005)


def load_labels(uid: str) -> pd.DataFrame:
    path = Path("../data/raw") / uid / "labels.txt"
    if not path.exists():
        return pd.DataFrame(columns=["start", "end", "mode"])
    lab = pd.read_csv(path, sep="\t", skiprows=1, header=None, names=["start", "end", "mode"])
    for c in ("start", "end"):
        lab[c] = pd.to_datetime(lab[c], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    return lab.dropna()


# đặc trưng để chia nhóm
labels = {uid: load_labels(uid) for uid in stays["user_id"].unique()}
feat = stays.copy()
feat["observed_share"] = feat["observed_min"] / feat["duration_min"]
feat["vehicle_share"] = 0.0
for uid, idx in feat.groupby("user_id").groups.items():
    lab = labels[uid]
    lab = lab[lab["mode"].isin(VEHICLES)]
    if lab.empty:
        continue
    s = feat.loc[idx]
    best = np.zeros(len(s))
    for t0, t1 in zip(lab["start"].to_numpy(), lab["end"].to_numpy(), strict=True):
        ov = (np.minimum(s["departure"].to_numpy(), t1) - np.maximum(s["arrival"].to_numpy(), t0)) / np.timedelta64(1, "m")
        best = np.maximum(best, np.clip(ov, 0, None))
    feat.loc[idx, "vehicle_share"] = best / s["duration_min"].to_numpy()

local = {}
for uid in feat["user_id"].unique():
    p = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["datetime", "timestamp_local", "tz_name"])
    local[uid] = p.sort_values("datetime").reset_index(drop=True)


def to_local(uid: str, t: pd.Timestamp) -> pd.Timestamp:
    p = local[uid]
    return p["timestamp_local"].iloc[np.searchsorted(p["datetime"].to_numpy(), np.datetime64(t))]


feat["arrival_local"] = [to_local(u, t) for u, t in zip(feat["user_id"], feat["arrival"], strict=True)]
feat["departure_local"] = [to_local(u, t) for u, t in zip(feat["user_id"], feat["departure"], strict=True)]
three_am = feat["arrival_local"].dt.normalize() + pd.Timedelta(hours=3)          # 03:00 đầu tiên kể từ giờ đến
three_am = three_am.where(three_am >= feat["arrival_local"], three_am + pd.Timedelta(days=1))
feat["overnight"] = three_am <= feat["departure_local"]

masks = {
    "random": pd.Series(True, index=feat.index),
    "short": feat["duration_min"].between(30, 40),
    "long": feat["duration_min"] > 12 * 60,
    "sparse": feat["observed_share"] < 0.10,
    "overnight": feat["overnight"],
    "in_vehicle_label": feat["vehicle_share"] >= 0.8,
}
picked, strata_of = [], {}
for name, n in STRATA.items():
    pool = feat.index[masks[name] & ~feat.index.isin(picked)]
    chosen = rng.choice(pool, size=min(n, len(pool)), replace=False)
    picked += list(chosen)
    strata_of.update({i: name for i in chosen})
print("Pool sizes:", {k: int(m.sum()) for k, m in masks.items()})

order = rng.permutation(picked)                    # thứ tự chấm xáo trộn, không lộ nhóm
sample_rows, items = [], []
for k, i in enumerate(order, start=1):
    r = feat.loc[i]
    sid = f"s{k:02d}"
    p = pd.read_parquet(USERS_DIR / f"user_{r.user_id}.parquet", columns=["datetime", "lat", "lon"]).sort_values("datetime")
    w = p[(p["datetime"] >= r.arrival - CONTEXT) & (p["datetime"] <= r.departure + CONTEXT)]
    if len(w) > MAX_POINTS:
        w = w.iloc[np.unique(np.linspace(0, len(w) - 1, MAX_POINTS).round().astype(int))]
    phase = np.where(w["datetime"] < r.arrival, 0, np.where(w["datetime"] > r.departure, 2, 1))
    dx = (w["lon"].to_numpy() - r.lon) * np.cos(np.radians(r.lat)) * 111_320
    dy = (w["lat"].to_numpy() - r.lat) * 110_540
    tmin = (w["datetime"] - r.arrival).dt.total_seconds().to_numpy() / 60
    lab = labels[r.user_id]
    lab = lab[(lab["end"] >= r.arrival - CONTEXT) & (lab["start"] <= r.departure + CONTEXT)]
    items.append({
        "id": sid, "user": r.user_id, "lat": round(r.lat, 6), "lon": round(r.lon, 6),
        "arrival_local": f"{r.arrival_local:%Y-%m-%d %H:%M}", "departure_local": f"{r.departure_local:%Y-%m-%d %H:%M}",
        "tz": local[r.user_id]["tz_name"].iloc[0], "duration_min": round(r.duration_min, 1), "observed_min": round(r.observed_min, 1),
        "num_points": int(r.num_points),
        "labels": [{"mode": m, "start": round((a - r.arrival).total_seconds() / 60, 1), "end": round((b - r.arrival).total_seconds() / 60, 1)}
                   for a, b, m in zip(lab["start"], lab["end"], lab["mode"], strict=True)],
        "points": [[round(float(x), 1), round(float(y), 1), round(float(t), 2), int(ph)] for x, y, t, ph in zip(dx, dy, tmin, phase, strict=True)],
    })
    sample_rows.append({"id": sid, "stratum": strata_of[i], "user_id": r.user_id, "arrival": r.arrival, "departure": r.departure,
                        "duration_min": r.duration_min, "observed_share": r.observed_share, "vehicle_share": r.vehicle_share})
audit_sample = pd.DataFrame(sample_rows)
audit_sample.to_csv(TABLE_DIR / "audit_sample.csv", index=False)
AUDIT_JSON.write_text(json.dumps({"stays": items}, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print(f"Audit sample: {len(items)} stays, {AUDIT_JSON.stat().st_size / 1e3:.0f} kB")
audit_sample["stratum"].value_counts()

Pool sizes: {'random': 18597, 'short': 2823, 'long': 1929, 'sparse': 9202, 'overnight': 3342, 'in_vehicle_label': 185}


Audit sample: 60 stays, 879 kB


stratum
random              30
long                 6
overnight            6
in_vehicle_label     6
sparse               6
short                6
Name: count, dtype: int64

### 3.1 Kết quả chấm
Kết quả được chấm trên trang "Chấm stay-point GeoLife" (claude.ai) và xuất từ cơ sở dữ liệu của trang ra `tables/audit_verdicts.csv`. **Độ chính xác** = đúng / (đúng + sai); "không chắc" được đếm riêng, không tính vào. Khoảng tin cậy 95% theo Wilson. Chỉ nhóm `random` ước lượng cho toàn bộ stay-point; các nhóm còn lại cho biết loại rủi ro nào thật sự sai.

In [7]:
VERDICTS_CSV = TABLE_DIR / "audit_verdicts.csv"


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre, half = (p + z * z / (2 * n)) / (1 + z * z / n), z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (centre - half, centre + half)


if VERDICTS_CSV.exists():
    graded = audit_sample.merge(pd.read_csv(VERDICTS_CSV), on="id", how="left")
    rows = []
    for name, g in [*graded.groupby("stratum", sort=False), ("all (not a population estimate)", graded)]:
        c, w, u = (int((g["verdict"] == v).sum()) for v in ("correct", "wrong", "unsure"))
        lo, hi = wilson(c, c + w)
        lo2, hi2 = wilson(c, c + w + u)
        rows.append({"stratum": name, "sampled": len(g), "graded": c + w + u, "correct": c, "wrong": w, "unsure": u,
                     "precision": c / (c + w) if c + w else np.nan, "ci95_low": lo, "ci95_high": hi,
                     "precision_unsure_as_wrong": c / (c + w + u) if c + w + u else np.nan, "ci95_low_unsure_as_wrong": lo2,
                     "ci95_high_unsure_as_wrong": hi2})
    audit_result = pd.DataFrame(rows)
    audit_result.to_csv(TABLE_DIR / "audit_result.csv", index=False)
    print(graded.loc[graded["verdict"] != "correct", ["id", "stratum", "user_id", "arrival", "duration_min", "verdict", "reasons", "note"]].to_string())
    rnd = graded[graded["stratum"] == "random"]
    dwell = rnd.groupby("verdict")["duration_min"].sum() / rnd["duration_min"].sum()
    print("Random sample, share of dwell time by verdict:", dwell.round(3).to_dict(),
          "| median duration by verdict (min):", rnd.groupby("verdict")["duration_min"].median().round(0).to_dict())
else:
    audit_result = pd.DataFrame()
    print("Chưa có kết quả chấm (tables/audit_verdicts.csv).")
audit_result.round(3)

     id           stratum user_id             arrival  duration_min verdict        reasons                                                                                                                            note
21  s22            random     003 2009-05-31 09:31:31     42.350000   wrong  moving_slowly              theo trên vệ tính tôi thấy nó là 1 đường lớn đi. Nên tôi nghĩ giai đoạn ghi GPS này là đang kẹt xe hoặc thế nào đó
28  s29  in_vehicle_label     068 2009-08-24 11:31:56     30.166667   wrong  moving_slowly  cái này là stay-point do đúng ngưỡng, nhưng dựa trên nhãn được đánh là xe buýt thì đây là di chuyển chậm đón khách của xe buýt
33  s34             short     126 2008-12-14 12:07:44     39.533333  unsure  moving_slowly                                                                                                                             NaN
34  s35  in_vehicle_label     020 2011-09-11 07:07:45     30.300000   wrong  moving_slowly                                  

,stratum,sampled,graded,correct,wrong,unsure,precision,ci95_low,ci95_high,precision_unsure_as_wrong,ci95_low_unsure_as_wrong,ci95_high_unsure_as_wrong
0,long,6,6,6,0,0,1.000,0.610,1.000,1.000,0.610,1.000
1,random,30,30,23,2,5,0.920,0.750,0.978,0.767,0.591,0.882
2,overnight,6,6,5,0,1,1.000,0.566,1.000,0.833,0.436,0.970
3,in_vehicle_label,6,6,3,3,0,0.500,0.188,0.812,0.500,0.188,0.812
4,sparse,6,6,6,0,0,1.000,0.610,1.000,1.000,0.610,1.000
5,short,6,6,4,1,1,0.800,0.376,0.964,0.667,0.300,0.903
6,all (not a population estimate),60,60,47,6,7,0.887,0.774,0.947,0.783,0.664,0.869


## Kết luận tầng 3

**Cả 18,597 stay-point của 174 user thoả đủ 9 tính chất, 0 vi phạm.** Không có điểm nào bị kéo vào cửa sổ khi đang cách tâm hơn 200 m (xa nhất là 199.998 m). Không stay-point nào vắt qua khoảng ngắt > 18 h hay chồng lên stay-point khác. Mọi giờ đến, giờ đi, tâm, số điểm và `observed_minutes` đều tính lại khớp từ chính các điểm GPS.

**Bộ kiểm tra có khả năng bắt lỗi (đối chứng ở mục 2.1):**

| Cách làm hỏng | Bị bắt |
|---|---|
| Sửa trực tiếp kết quả: dịch tâm 1 m, số điểm + 1, giờ đến − 1 giây, `observed` + 1 phút, thời lượng + 1 phút | **100%** (886/886 mỗi loại) |
| Nhân đôi một dòng | 1/1 |
| Bỏ việc cắt tại khoảng ngắt > 18 h | **42/42** |
| Detector bán kính 300 m | 1,010/1,017 (99%) |
| Detector bán kính 150 m | 646/753 (86%) |

Với "giờ đến − 1 giây", tính chất nhắm tới chỉ bắt 733/886. Ở dữ liệu ghi mỗi giây, giờ mới vẫn trùng với một điểm thật, nhưng số điểm và tâm khi đó không còn khớp, nên tổng cộng vẫn bắt đủ 886/886.

**Giới hạn:** các stay-point lọt qua ở hai dòng 300 m và 150 m **không sai theo định nghĩa**. Chúng là những cửa sổ hợp lệ 200 m / 30 phút, chỉ bắt đầu ở điểm khác vì lượt quét trước đó đã đi theo hướng khác. Các tính chất ở đây kiểm **từng stay-point có hợp lệ không**. Chúng không kiểm **có bỏ sót stay-point nào không**. Phần bỏ sót được đo ở notebook 07 (độ phủ các khoảng dừng giữa hai chuyến đi có nhãn) và sẽ được kiểm thêm bằng mắt ở tầng 4.

**Đã đưa vào test:** cùng các phép kiểm chạy trong `tests/integration/test_pipeline_on_real_data.py::test_stay_points_satisfy_detector_definition` trên dữ liệu thật của user 010. Mọi thay đổi sau này ở detector hay pipeline mà phá vỡ một tính chất sẽ bị test báo.

## Kết luận tầng 4 (chấm mù 60 stay-point, mục 3)

| Nhóm | Chấm | Đúng | Sai | Không chắc | Độ chính xác (95% CI) | Nếu coi "không chắc" là sai |
|---|---|---|---|---|---|---|
| **random** (đại diện cho toàn bộ) | 30 | 23 | 2 | 5 | **92%** (75–98%) | 77% (59–88%) |
| short (30–40 phút) | 6 | 4 | 1 | 1 | 80% | 67% |
| long (> 12 h) | 6 | 6 | 0 | 0 | 100% | 100% |
| sparse (có điểm < 10% thời gian) | 6 | 6 | 0 | 0 | 100% | 100% |
| overnight (có mặt lúc 03:00) | 6 | 5 | 0 | 1 | 100% | 83% |
| in_vehicle_label (stay-point "giả" theo notebook 07) | 6 | 3 | 3 | 0 | 50% | 50% |

- **Độ chính xác của bộ phát hiện nằm khoảng 77–92%.** Con số này tính theo số lượng stay-point và ước lượng từ nhóm ngẫu nhiên; cận dưới tương ứng với việc coi mọi ca không chắc là sai. Mẫu nhỏ nên khoảng tin cậy còn rộng.
- **Tính theo thời gian ở lại, sai số nhỏ hơn nhiều.** Trong nhóm ngẫu nhiên, 90.6% thời gian nằm trong các stay-point được chấm đúng, chỉ 2.0% chấm sai và 7.4% không chắc. Các stay-point sai hoặc không chắc đều ngắn (trung vị 43–45 phút, so với 113 phút ở stay-point đúng). Home/Office được tính theo thời gian, nên chịu ảnh hưởng ít.
- **Chỉ có một kiểu lỗi: di chuyển chậm.** Cả 6 ca sai và 5/7 ca không chắc có điểm trải dọc một con đường: kẹt xe, hoặc xe buýt chạy chậm đón khách. Đó là di chuyển ≥ 30 phút mà vẫn nằm trong 200 m. Không có ca nào thuộc kiểu "gộp hai nơi" hay "hầu như không có điểm".
- **Nối file (notebook 04) và giữ khoảng ngắt bên trong stay-point (notebook 06) không tạo ra lỗi trong mẫu.** Ba nhóm long, sparse và overnight cho 17 ca đúng, 1 ca không chắc, 0 ca sai. Riêng nhóm sparse (gần như toàn thời gian không có điểm) đúng cả 6.
- **Stay-point "giả" theo nhãn của notebook 07 chỉ sai một nửa.** 3/6 ca thật sự là xe đang chạy chậm. 3/6 ca còn lại là lúc dừng thật nằm bên trong một nhãn chuyến đi, ví dụ tàu hoả đứng chờ trên đường ray hơn 30 phút (s28). Điều này khớp với nhận định ở notebook 07 rằng chỉ số dựa trên nhãn đánh giá quá cao số stay-point giả.

**Quyết định:** giữ bộ phát hiện như hiện tại. Lỗi còn lại là di chuyển chậm, thường kéo dài 30–45 phút trên đường lớn. Có thể lọc kiểu lỗi này bằng một tiêu chí hình dạng, ví dụ cụm điểm trải dài theo một hướng. Tuy nhiên tiêu chí đó cần ngưỡng và bằng chứng riêng, nên sẽ được cân nhắc khi classifier cho thấy lỗi này thật sự ảnh hưởng tới Home/Office.

**Giới hạn:**
- Chỉ có **một người chấm**, nên chưa đo được mức đồng thuận giữa nhiều người chấm.
- Mỗi nhóm rủi ro chỉ có 6 ca, nên chỉ cho thấy xu hướng.
- Ảnh vệ tinh chụp sau năm 2020, trong khi dữ liệu GeoLife ghi năm 2007–2012.
- Với user có nhãn, người chấm nhìn thấy nhãn chuyến đi, nên ở nhóm in_vehicle_label kết luận có thể bị nhãn ảnh hưởng.